# 01 · Qualidade do bronze — cadastro e comercial

Primeira varredura de qualidade da camada bronze, cobrindo o domínio cadastral (organizações, itens, endereços, tarifas e parametrização). O método: cada seção faz a pergunta em **SQL (DuckDB sobre parquet)**, explora o resultado em **pandas** e fecha com uma **Nota Técnica** (Observado · Por que importa · Ação proposta). Nenhum número é digitado: tudo que este notebook afirma sai das células executadas.

**Como rodar:** com o bronze extraído (`uv run bronze-staging`), abra este notebook com o kernel da `.venv` do projeto (`uv sync` cria). Os caminhos são resolvidos pela configuração do projeto; não há caminho absoluto aqui.

In [1]:
import pandas as pd

from logistica_fictitur.lake import conectar

pd.set_option("display.max_rows", 60)
conn = conectar()


def sql(consulta: str) -> pd.DataFrame:
    """A ponte do notebook: SQL processa os milhões, pandas recebe o resumo."""
    return conn.execute(consulta).df()

## 1. Panorama do domínio

Antes de julgar qualidade, medir o terreno: quantas linhas cada tabela cadastral tem. Paralelo SQL → pandas: o laço abaixo faz o papel de um `UNION ALL` de `COUNT(*)`, e o `DataFrame` final é a "tabela de resultado".

In [2]:
views = sql("SELECT view_name FROM duckdb_views() WHERE NOT internal")
cadastrais = sorted(v for v in views["view_name"] if v.startswith("bronze_cadastro_"))
panorama = pd.DataFrame(
    {
        "tabela": [v.removeprefix("bronze_cadastro_") for v in cadastrais],
        "linhas": [conn.execute(f'SELECT count(*) FROM "{v}"').fetchone()[0] for v in cadastrais],
    }
).sort_values("linhas", ascending=False, ignore_index=True)
panorama

,tabela,linhas
0,endereco,19737
1,item,6804
2,cota,5286
3,posicao_armazem,4640
4,organizacao,240
5,sla_manuseio,203
6,representante,170
7,transportador,130
8,doca,96
9,sla_transporte,81


> **Nota Técnica 1**  
>**Observado:** o domínio tem uma tabela completamente vazia (`meta_frete`) e as demais povoadas em escalas coerentes com o negócio (endereços na casa das dezenas de milhares, catálogos na casa das dezenas).
> - **Por que importa:** tabela vazia em cadastro costuma significar funcionalidade nunca implantada ou carga quebrada; qualquer indicador que dependa dela nascerá nulo em silêncio.
> - **Ação proposta:** registrar `meta_frete` como achado; a silver não a inventa (ausência não se preenche), e o time de origem é quem responde se há carga faltando.

## 2. Completude — onde faltam valores

O `SUMMARIZE` do DuckDB perfila uma tabela inteira numa instrução (o equivalente a escrever um `COUNT(*) FILTER (WHERE col IS NULL)` por coluna). Abaixo, o perfil do catálogo de itens, ordenado pelo percentual de nulos.

In [3]:
perfil_item = sql("SUMMARIZE SELECT * FROM bronze_cadastro_item")
colunas = ["column_name", "column_type", "null_percentage", "approx_unique", "min", "max"]
perfil_item[colunas].sort_values("null_percentage", ascending=False, ignore_index=True)

,column_name,column_type,null_percentage,approx_unique,min,max
0,url_foto,VARCHAR,100.00,0,NaN,NaN
1,classe_anvisa,VARCHAR,100.00,0,NaN,NaN
2,material,VARCHAR,100.00,0,NaN,NaN
3,estrategia,VARCHAR,100.00,0,NaN,NaN
4,valor_unitario,"DECIMAL(12,2)",1.65,5826,4.92,971.60
5,id,BIGINT,0.00,6424,1,6804
6,cod_item,VARCHAR,0.00,6884,AAV00001,XNG00019
7,organizacao_id,BIGINT,0.00,225,38,240
8,marca,VARCHAR,0.00,12,Alva,Vivace
9,subgrupo,VARCHAR,0.00,27,Achocolatados,Teclados


In [4]:
sql("""
    SELECT count(*) AS itens,
           count(*) FILTER (WHERE valor_unitario IS NULL) AS sem_valor_unitario,
           round(100.0 * count(*) FILTER (WHERE valor_unitario IS NULL) / count(*), 2)
               AS pct_sem_valor
    FROM bronze_cadastro_item
""")

,itens,sem_valor_unitario,pct_sem_valor
0,6804,112,1.65


> **Nota Técnica 2**  
>**Observado:** três grupos de lacuna no catálogo de itens: colunas 100% nulas (`url_foto`, `material`, `classe_anvisa`, `estrategia`), que nunca foram preenchidas na origem; uma fração pequena mas perigosa de itens **sem valor unitário**; e nulos estruturais esperados (ex.: `cod_item_cliente` em parte do catálogo).
> - **Por que importa:** item sem valor unitário é **furo de cobertura fiscal**: qualquer soma de valor de estoque ou de pedido que o envolva subestima em silêncio; colunas 100% nulas poluem o modelo e enganam quem monta relatório.
> - **Ação proposta:** silver **marca** o furo (`fl_sem_valor_unitario`) sem inventar preço (ausência não se preenche); colunas 100% nulas ficam documentadas e de fora das camadas analíticas.

## 3. Validade — valores fora do domínio

Formato de documento e faixas contratuais: CNPJ precisa de 14 dígitos, CEP de 8, e o OTIF contratual de um cliente vive entre 0 e 1.

In [5]:
sql("""
    SELECT 'organizacao.cnpj com formato invalido' AS verificacao,
           count(*) FILTER (WHERE cnpj !~ '^[0-9]{14}$') AS ocorrencias
    FROM bronze_cadastro_organizacao
    UNION ALL
    SELECT 'endereco.cep com formato invalido',
           count(*) FILTER (WHERE cep IS NOT NULL AND cep !~ '^[0-9]{8}$')
    FROM bronze_cadastro_endereco
    UNION ALL
    SELECT 'organizacao.otif_contratual fora de [0,1]',
           count(*) FILTER (WHERE otif_contratual IS NOT NULL
                            AND otif_contratual NOT BETWEEN 0 AND 1)
    FROM bronze_cadastro_organizacao
    UNION ALL
    SELECT 'item com dimensao fisica nao positiva',
           count(*) FILTER (WHERE peso_kg <= 0 OR altura_cm <= 0
                            OR largura_cm <= 0 OR comprimento_cm <= 0)
    FROM bronze_cadastro_item
""")

,verificacao,ocorrencias
0,organizacao.cnpj com formato invalido,0
1,endereco.cep com formato invalido,0
2,"organizacao.otif_contratual fora de [0,1]",0
3,item com dimensao fisica nao positiva,0


> **Nota Técnica 3**  
> **Observado:** os formatos de documento e as faixas contratuais do cadastro passaram limpos nesta base.
> - **Por que importa:** validade zero aqui não aposenta a verificação: ela é barata e fica de sentinela para cargas futuras (o dia em que um CNPJ vier com máscara, este notebook acusa).
> - **Ação proposta:** promover estas checagens à régua permanente da silver, mesmo hoje sem ocorrências.

## 4. Unicidade — o que não pode se repetir

Chaves de negócio do cadastro: sigla e CNPJ de organização, código de item dentro do cliente.

In [6]:
sql("""
    SELECT 'organizacao.sigla duplicada' AS verificacao,
           count(*) - count(DISTINCT sigla) AS excedentes
    FROM bronze_cadastro_organizacao
    UNION ALL
    SELECT 'organizacao.cnpj duplicado',
           count(*) - count(DISTINCT cnpj)
    FROM bronze_cadastro_organizacao
    UNION ALL
    SELECT 'item.cod_item duplicado dentro do cliente',
           count(*) - count(DISTINCT (organizacao_id, cod_item))
    FROM bronze_cadastro_item
    UNION ALL
    SELECT 'endereco de entrega duplicado (cnpj_local + logradouro + numero)',
           count(*) - count(DISTINCT (cnpj_local, logradouro, numero))
    FROM bronze_cadastro_endereco WHERE tipo = 'ENTREGA'
""")

,verificacao,excedentes
0,organizacao.sigla duplicada,0
1,organizacao.cnpj duplicado,0
2,item.cod_item duplicado dentro do cliente,0
3,endereco de entrega duplicado (cnpj_local + lo...,0


> **Nota Técnica 4**  
> **Observado:** todas as chaves de negócio verificadas estão íntegras, incluindo a identidade dos endereços de entrega (nenhuma repetição de CNPJ + logradouro + número).
> - **Por que importa:** unicidade limpa em cadastro é raridade bem-vinda; e a verificação registrada continua de sentinela, porque endereço duplicado é o defeito que mais aparece quando cadastros crescem por digitação.
> - **Ação proposta:** nenhuma correção; a checagem entra na régua permanente da silver.

## 5. Consistência — as pontas amarram?

Verificações de referência entre tabelas do domínio (anti-joins: o `LEFT JOIN ... WHERE alvo IS NULL` que todo teste de vaga adora) e uma regra de negócio do caso: **galpão só pertence a matriz ou base, nunca a cliente**.

In [7]:
sql("""
    SELECT 'item orfao de organizacao' AS verificacao, count(*) AS ocorrencias
    FROM bronze_cadastro_item i
    LEFT JOIN bronze_cadastro_organizacao o ON o.id = i.organizacao_id
    WHERE o.id IS NULL
    UNION ALL
    SELECT 'endereco orfao de municipio', count(*)
    FROM bronze_cadastro_endereco e
    LEFT JOIN bronze_cadastro_municipio m ON m.id = e.municipio_id
    WHERE m.id IS NULL
    UNION ALL
    SELECT 'galpao pertencendo a CLIENTE (regra do caso)', count(*)
    FROM bronze_cadastro_galpao g
    JOIN bronze_cadastro_organizacao o ON o.id = g.organizacao_id
    WHERE o.tipo_parceria = 'CLIENTE'
    UNION ALL
    SELECT 'tarifa de armazenagem de organizacao inexistente', count(*)
    FROM bronze_financeiro_tarifa_armazenagem t
    LEFT JOIN bronze_cadastro_organizacao o ON o.id = t.organizacao_id
    WHERE o.id IS NULL
""")

,verificacao,ocorrencias
0,item orfao de organizacao,0
1,endereco orfao de municipio,0
2,galpao pertencendo a CLIENTE (regra do caso),0
3,tarifa de armazenagem de organizacao inexistente,0


> **Nota Técnica 5**  
> **Observado:** as referências do domínio cadastral amarram sem órfãos, e a regra física do caso (galpões só de matriz e bases) se confirma nos dados.
> - **Por que importa:** consistência aqui é o alicerce de todos os joins das próximas camadas; órfão em cadastro vira linha perdida em fato.
> - **Ação proposta:** nenhuma correção; as checagens entram na régua permanente.

## 6. Consolidação dos achados do domínio

O resumo executável da varredura: cada achado com sua medida, recalculado aqui (nunca copiado à mão). Este quadro alimenta o catálogo de achados e regras da silver.

In [ ]:
achados = sql("""
    SELECT 'CAD-01' AS codigo,
           'meta_frete sem nenhuma linha' AS achado,
           (SELECT count(*) FROM bronze_cadastro_meta_frete) AS medida,
           'documentar; ausencia nao se preenche' AS acao_proposta
    UNION ALL
    SELECT 'CAD-02', 'itens sem valor unitario (furo fiscal)',
           (SELECT count(*) FROM bronze_cadastro_item WHERE valor_unitario IS NULL),
           'marcar com flag na silver; nao inventar preco'
    UNION ALL
    SELECT 'CAD-03', 'colunas 100% nulas no catalogo de itens',
           (SELECT count(*) FROM (SUMMARIZE SELECT * FROM bronze_cadastro_item)
            WHERE null_percentage = 100),
           'documentar e excluir das camadas analiticas'
""")
achados

,codigo,achado,medida,acao_proposta
0,CAD-01,meta_frete sem nenhuma linha,0,documentar; ausencia nao se preenche
1,CAD-02,itens sem valor unitario (furo fiscal),112,marcar com flag na silver; nao inventar preco
2,CAD-03,colunas 100% nulas no catalogo de itens,4,documentar e excluir das camadas analiticas


: 

> **Nota Técnica 6 (fecho)**  
> **Observado:** três achados no domínio cadastral, nenhum de correção destrutiva: um vazio estrutural (`meta_frete`), um furo fiscal a sinalizar (itens sem valor unitário) e colunas mortas a documentar; validade, unicidade e consistência passaram limpas, e as checagens ficam de sentinela.
> - **Por que importa:** é o retrato de um cadastro real: o problema raramente é o erro escandaloso, é a lacuna silenciosa.
> - **Ação proposta:** levar CAD-01..03 ao catálogo de achados da silver para aprovação das regras antes de qualquer transformação. Próxima varredura: **02 · pedidos e entregas**, onde vivem as datas-sentinela e os prazos.